# Benchmark Giám Khảo VCuboidFIT — Colab A100

**Câu hỏi:** Lô ảnh VCuboidFIT có làm các giám khảo AEB độc lập bỏ sót nhiều đối tượng thật hơn bên thứ 3 không?

**4 luật bất biến (SPEC-P04 §2):**
1. **Zero-shot:** giám khảo JA/JB không được fine-tune trên dữ liệu dự án.
2. **2 giám khảo độc lập:** JA (FasterRCNN) và JB (RT-DETR) từ hai họ model khác nhau.
3. **Mở POOL một lần:** `selections_sha256` phải commit trước khi chạy ô 7.
4. **Prereg:** khối prereg phải băm khớp trước khi chạy bất kỳ bước chấm nào.


In [ ]:
# Ô 1 — Cấu hình (≈ 30 giây đọc hiểu) — sửa tại đây, không sửa ô khác
# @param ["nuscenes-pilot", "pandaset", "av2"] {type:"string"}
DATASET = "nuscenes-pilot"

DRIVE_ROOT = "MyDrive/vcf_judge"   # thư mục gốc trên Drive
REPO_SHA   = ""                    # SHA commit code đã push; bắt buộc điền

# SHA prereg — lấy từ evidence/PREREG/prereg_sha256.txt
EXPECTED_PREREG = "4b8ef249c1d774c3dd58694a599e3c782c05949a3b8aa64201084c67f8c252db"

# Manifest gói dữ liệu (JSON) trên Drive
PACK_MANIFEST = f"{DRIVE_ROOT}/{DATASET}/manifest.json"

# Mặc định an toàn: chỉ chạy tới CAL + power; người bật tay để mở POOL/EVAL
RUN_POOL  = False   # @param {type:"boolean"}
RUN_TRAIN = False   # @param {type:"boolean"}

JOB_DIR = f"/content/vcf_jobs/{DATASET}"
# Gói dữ liệu giải nén vào JOB_DIR: data-root ở JOB_DIR/data_root, config engine đóng băng ở JOB_DIR/final_config.json
DATA_ROOT = f"{JOB_DIR}/data_root"
FINAL_CONFIG = f"{JOB_DIR}/final_config.json"


In [ ]:
# Ô 2 — Mount Drive, giải nén code, cài worker, kiểm GPU (~3–5 phút)
import sys, time, subprocess, zipfile, pathlib
_t0 = time.perf_counter()

from google.colab import drive
drive.mount("/content/drive")
DRIVE_ABS = f"/content/drive/{DRIVE_ROOT}"
assert REPO_SHA, "Điền REPO_SHA ở ô 1"

# Giải nén code ZIP đúng REPO_SHA rồi cài worker TRƯỚC khi import c4
code_zip = pathlib.Path(f"{DRIVE_ABS}/code_{REPO_SHA}.zip")
if not code_zip.is_file():
    raise FileNotFoundError(f"Không tìm thấy: {code_zip}")
with zipfile.ZipFile(code_zip) as zf:
    zf.extractall("/content/vcf_repo")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                "/content/vcf_repo/model/worker[gpu,bench,judge]"], check=True)
sys.path.insert(0, "/content/vcf_repo/model/worker")
print(f"✅ Code {REPO_SHA} đã cài")

from c4.bench.colab import check_gpu, append_timing
gpu_name = check_gpu(warn_non_a100=True)
import torch, torchvision, transformers
print(f"torch={torch.__version__}, torchvision={torchvision.__version__}, transformers={transformers.__version__}")
append_timing(DRIVE_ABS, "setup", time.perf_counter() - _t0, status="ok")


In [ ]:
# Ô 3 — Ghép gói dữ liệu + kiểm SHA (thời gian tùy cỡ gói)
import time
_t0 = time.perf_counter()

from c4.bench.colab import assemble_pack, append_timing
assemble_pack(
    manifest_path=f"{DRIVE_ABS}/{DATASET}/manifest.json",
    pack_dir=f"{DRIVE_ABS}/{DATASET}/",
    out_dir=JOB_DIR,
)

append_timing(DRIVE_ABS, "assemble_pack", time.perf_counter() - _t0, status="ok")


In [ ]:
# Ô 4 — Bảng trạng thái cổng (xanh/đỏ, ~vài giây)
from c4.bench.colab import print_gate_status
print_gate_status(JOB_DIR, EXPECTED_PREREG)


In [ ]:
# Ô 5 — Chuẩn bị engine + bên thứ 3, chạy select (~20–60 phút lần đầu)
import time
_t0 = time.perf_counter()

from c4.bench.colab import run_bench_judge, append_timing, get_vram_mb
import pathlib

sel_done = (pathlib.Path(JOB_DIR) / "bench" / "selections.json").is_file()
if sel_done:
    print("ℹ️  selections.json đã có — bỏ qua ô 5.")
else:
    run_bench_judge(
        "select",
        "--job-dir", JOB_DIR,
        "--methods", "OURS,M1,M2,X0,X1,X2,X3,N1,N2",
        "--dataset", DATASET, "--data-root", DATA_ROOT, "--final-config", FINAL_CONFIG,
        drive_root=DRIVE_ABS,
        step="select",
    )
    append_timing(DRIVE_ABS, "select_total", time.perf_counter() - _t0,
                  vram_mb=get_vram_mb(), status="ok")


In [ ]:
# Ô 6 — Judge CAL + power.json (τ, validity, MDE) (~10–20 phút)
import time
_t0 = time.perf_counter()

from c4.bench.colab import run_bench_judge, append_timing, get_vram_mb
import pathlib

calib_done = (pathlib.Path(JOB_DIR) / "bench" / "judge" / "judge_calib.json").is_file()
if calib_done:
    print("ℹ️  judge_calib.json đã có — bỏ qua judge CAL.")
else:
    run_bench_judge(
        "judge", "--split", "cal", "--judges", "JA,JB",
        "--job-dir", JOB_DIR, "--data-root", DATA_ROOT,
        drive_root=DRIVE_ABS,
        step="judge_cal",
    )

# Exploratory select CAL → power.json
run_bench_judge(
    "select", "--split", "cal", "--exploratory",
    "--job-dir", JOB_DIR,
    "--methods", "OURS,M1,M2,X0,X1,X2,X3,N1,N2",
    "--dataset", DATASET, "--data-root", DATA_ROOT, "--final-config", FINAL_CONFIG,
    drive_root=DRIVE_ABS,
    step="select_cal_exploratory",
)

import json
power = json.loads((pathlib.Path(JOB_DIR) / "bench" / "power.json").read_text())
print("📊 Giám khảo + power:", json.dumps(power, indent=2, ensure_ascii=False))
append_timing(DRIVE_ABS, "judge_cal_total", time.perf_counter() - _t0,
              vram_mb=get_vram_mb(), status="ok")


In [ ]:
# Ô 7 — Judge POOL — chỉ chạy khi RUN_POOL=True (~10–40 phút)
import time
_t0 = time.perf_counter()

from c4.bench.colab import run_bench_judge, append_timing, get_vram_mb

if RUN_POOL:
    # marker ghi TRƯỚC lệnh (open_once trong CLI)
    run_bench_judge(
        "judge", "--split", "pool", "--judges", "JA,JB",
        "--job-dir", JOB_DIR, "--data-root", DATA_ROOT,
        drive_root=DRIVE_ABS,
        step="judge_pool",
    )
    append_timing(DRIVE_ABS, "judge_pool_total", time.perf_counter() - _t0,
                  vram_mb=get_vram_mb(), status="ok")
else:
    print("ℹ️  RUN_POOL=False — bỏ qua. Bật trong ô 1 để chạy.")


In [ ]:
# Ô 8 — bench_judge report → report_judge.json, bench_table.md (~vài phút)
import time
_t0 = time.perf_counter()

from c4.bench.colab import run_bench_judge, append_timing

run_bench_judge(
    "report",
    "--job-dir", JOB_DIR, "--data-root", DATA_ROOT,
    drive_root=DRIVE_ABS,
    step="report",
)
append_timing(DRIVE_ABS, "report", time.perf_counter() - _t0, status="ok")


In [ ]:
# Ô 9 — Pha 4: train-eval (~5 giờ AV2) — chỉ chạy khi RUN_TRAIN=True
import time
_t0 = time.perf_counter()

from c4.bench.colab import run_bench_judge, append_timing, get_vram_mb

if RUN_TRAIN:
    run_bench_judge(
        "train-eval",
        "--job-dir", JOB_DIR,
        drive_root=DRIVE_ABS,
        step="train_eval",
    )
    append_timing(DRIVE_ABS, "train_eval_total", time.perf_counter() - _t0,
                  vram_mb=get_vram_mb(), status="ok")
else:
    print("ℹ️  RUN_TRAIN=False — bỏ qua.")


In [ ]:
# Ô 10 — Vẽ dashboard inline (~giây)
import json, pathlib
from c4.bench.dashboard import render

report_path = pathlib.Path(JOB_DIR) / "bench" / "report" / "report_judge.json"
train_path  = pathlib.Path(JOB_DIR) / "bench" / "report" / "report_train.json"

if not report_path.is_file():
    print("⚠️  report_judge.json chưa có — bỏ qua dashboard.")
else:
    report_judge = json.loads(report_path.read_text(encoding="utf-8"))
    report_train = json.loads(train_path.read_text(encoding="utf-8")) if train_path.is_file() else None
    out_dir = pathlib.Path(JOB_DIR) / "bench" / "report"
    html_path = render(report_judge, report_train, pathlib.Path(JOB_DIR), out_dir)

    # Hiển thị inline
    from IPython.display import HTML, display  # Colab không phục vụ file /content qua IFrame
    display(HTML(pathlib.Path(html_path).read_text(encoding="utf-8")))
    print(f"✅ Dashboard: {html_path}")


In [ ]:
# Ô 11 — Đóng gói results_bundle.zip lên Drive (~vài phút)
import time
_t0 = time.perf_counter()

from c4.bench.colab import pack_results_bundle, append_timing
bundle_path = pack_results_bundle(JOB_DIR, DRIVE_ABS, DATASET)
print(f"✅ Bundle: {bundle_path}")
append_timing(DRIVE_ABS, "pack_bundle", time.perf_counter() - _t0, status="ok")
